# Add Daily Demand

In [ ]:
# merge all_nodes demand data into 1 dataframe
# join using node id and area

In [ ]:
import folium

In [ ]:
# Create a base map
# You might want to center the map based on the data's extent
# For now, let's use a general center point (e.g., approximate center of Israel)
map_center = [31.77, 35.22] # Approximate center of Israel
m = folium.Map(location=map_center, zoom_start=8)

In [ ]:
# Add the gdf as a GeoJson layer to the map
# Ensure gdf is in WGS84 (EPSG:4326) for folium
# Use gdf_cleaned which has the correct columns after processing
gdf_wgs84 = gdf.to_crs(epsg=4326)

# Calculate centroids
gdf_wgs84['centroid'] = gdf_wgs84.geometry.centroid

# Create a FeatureGroup to add markers to
marker_group = folium.FeatureGroup(name='Hub Centroids')

# Add markers for each centroid
for index, row in gdf_wgs84.iterrows():
    if row['centroid'] is not None: # Check if centroid calculation was successful
        # Create a Tooltip
        tooltip_text = f"Group: {row['group']}"
        tooltip = folium.Tooltip(tooltip_text)

        folium.Marker(
            location=[row['centroid'].y, row['centroid'].x],
            tooltip=tooltip # Add the tooltip
        ).add_to(marker_group)

# Add the marker group to the map
marker_group.add_to(m)

# Add layer control to toggle centroid markers
folium.LayerControl().add_to(m)

In [ ]:

# Display the map
m

In [ ]:
xl = pd.ExcelFile('/content/drive/MyDrive/Hubs/Nodes_w_results_21082025.xlsx')

In [ ]:
xl.sheet_names

In [ ]:
sheet_names = {'Haifa': 'Haifa',
 'TelAviv': 'Tel Aviv',
 'BeerSheva': 'Beer Sheva',
 'Hadera': 'Hadera',
 'Jerusalem': 'Jerusalem',
 'HaifaMetronit': 'Haifa Metronit',
 'Ashkelon': 'Ashdod-Ashkelon',
               'National': 'Rail'}

In [ ]:
demand = {sheet: pd.read_excel(xl, sheet) for sheet in xl.sheet_names}

In [ ]:
# Iterate over a copy of the keys to avoid the RuntimeError
for key in list(demand.keys()):
  if key in sheet_names.keys():
    demand[sheet_names[key]] = demand.pop(key)

In [ ]:
for key in list(demand.keys()):
  if key not in sheet_names.values():
    demand.pop(key)

In [ ]:
demand.keys()

In [ ]:
demand['Beer Sheva'].rename(columns={'NODE_ID': 'Node'}, inplace=True)
demand['Beer Sheva']['TotalTransfers'] = 0

In [ ]:
demand['Beer Sheva'].head(1)

In [ ]:
demand['Hadera'].head(1)

In [ ]:
demand['Hadera'].rename(columns={'NodeID': 'Node',
                                 'On': 'Boardings_Daily',
                                 'Off': 'Alightings_Daily'},
                        inplace=True)
demand['Hadera']['TotalTransfers'] = 0
drop_cols = ['hub_name']
demand['Hadera'].drop(columns=drop_cols, inplace=True)

In [ ]:
demand['Hadera'].head(1)

In [ ]:
demand['Tel Aviv'].head(1)

In [ ]:
demand['Tel Aviv'].rename(columns={'TotalBoardings': 'Boardings_Daily',
                                   'TotalAlight': 'Alightings_Daily'},
                                   inplace=True)
drop_cols = ['NumLines','InitialBoardings','TransferBoardings','FinalAlight','TransferAlight','Unnamed: 9']
demand['Tel Aviv'].drop(columns=drop_cols, inplace=True)

In [ ]:
demand['Tel Aviv'].head(1)

In [ ]:
demand['Ashdod-Ashkelon'].head(1)

In [ ]:
demand['Ashdod-Ashkelon'].rename(columns={'InitialBoardings': 'Boardings_Daily',
                                      'FinalAlight': 'Alightings_Daily'},
                             inplace=True)
demand['Ashdod-Ashkelon']['TotalTransfers'] = 0
demand['Ashdod-Ashkelon'].head(1)

In [ ]:
demand['Haifa'].head(1)

In [ ]:
demand['Haifa'].rename(columns={'TotalBoardings': 'Boardings_Daily',
                                'TotalAlight': 'Alightings_Daily'},
                       inplace=True)
demand['Haifa']['TotalTransfers'] = demand['Haifa']['TransferBoardings'] + demand['Haifa']['TransferAlight']
drop_cols = ['InitialBoardings','TransferBoardings','FinalAlight','TransferAlight']
demand['Haifa'].drop(columns=drop_cols, inplace=True)

In [ ]:
demand['Haifa'].head(1)

In [ ]:
demand['Haifa Metronit'].head(1)

In [ ]:
demand['Haifa Metronit'].rename(columns={'ModelNode': 'Node'}, inplace=True)
demand['Haifa Metronit']['TotalTransfers'] = 0
drop_cols = ['NewNode','TotalDemand_Daily']
demand['Haifa Metronit'].drop(columns=drop_cols, inplace=True)

In [ ]:
demand['Haifa Metronit'].head(1)

In [ ]:
demand['Jerusalem'].head(1)

In [ ]:
demand['Jerusalem'].rename(columns={'ID': 'Node',
                                    'DailyBoard_2050': 'Boardings_Daily',
                                    'DailyAlight_2050': 'Alightings_Daily'},
                           inplace=True)
demand['Jerusalem']['TotalTransfers'] = 0
keep_cols = ['Node','Boardings_Daily','Alightings_Daily','TotalTransfers']
demand['Jerusalem'] = demand['Jerusalem'][keep_cols]

In [ ]:
demand['Jerusalem'].head(1)

In [ ]:
# add model source column
for key in list(demand.keys()):
  demand[key]['Model'] = key

In [ ]:
# before concatinating all the demand worksheets,
# need to add to each demand a column with its name ("hadera", "haifa" etc)
# need to rename boarding and alighting columns
# need to decide on a transfer columns for when there is one

In [ ]:
combined = pd.concat(demand.values(), ignore_index=True, axis=0)

In [ ]:
combined

In [ ]:
combined.to_csv('/content/drive/MyDrive/Hubs/demand_by_node_29102025.csv', encoding='utf-8', index=False)

In [ ]:
gdf['Model'].unique()

In [ ]:
combined['Model'].unique()

In [ ]:
# first - assign demand from the models: Haifa, Tel Aviv, Beer Sheva, Jerusalem, Ashdod-Ashkelon
# then, update demand for the Nodes in the models: Haifa Metronit, Hadera
# last, check the Netanya data, check national model data for specific locations, update Rail

In [ ]:
gdf.head(1)

In [ ]:
# create total demand column and transfers column
gdf['TotalDemand'] = 0
gdf['TotalTransfers'] = 0

In [ ]:
gdf.to_csv('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8', index=False)

In [ ]:
gdf = gpd.read_file('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8')

# Explicitly convert TotalDemand and TotalTransfers to numeric after loading
gdf['TotalDemand'] = pd.to_numeric(gdf['TotalDemand'], errors='coerce').fillna(0)
gdf['TotalTransfers'] = pd.to_numeric(gdf['TotalTransfers'], errors='coerce').fillna(0)
gdf['Line_Nunique'] = pd.to_numeric(gdf['Line_Nunique'], errors='coerce').fillna(0)

In [ ]:
gdf.head(5)

In [ ]:
gdf[gdf['node']=='511248']

In [ ]:
# update Haifa Metronit & Hadera

In [ ]:
demand['Hadera']

In [ ]:
gdf[gdf['node']==987294]

In [ ]:
# update Hadera using Hadera demand data

# Ensure 'Node' column in demand['Hadera'] is numeric
demand['Hadera']['Node'] = pd.to_numeric(demand['Hadera']['Node'], errors='coerce')

# Select relevant columns from demand['Hadera']
hadera_demand = demand['Hadera'][['Node', 'Boardings_Daily', 'Alightings_Daily', 'TotalTransfers']]

# Aggregate hadera_demand by Node, summing the demand and transfers
hadera_demand_agg = hadera_demand.groupby('Node').agg(
    Boardings_Hadera_sum=('Boardings_Daily', 'sum'),
    Alightings_Hadera_sum=('Alightings_Daily', 'sum'),
    TotalTransfers_Hadera_sum=('TotalTransfers', 'sum')
).reset_index() # Reset index to make 'Node' a column again

# Rename 'Node' to 'node' for matching with gdf
hadera_demand_agg.rename(columns={'Node': 'node'}, inplace=True)

# Ensure 'node' column in gdf is numeric for matching
gdf['node'] = pd.to_numeric(gdf['node'], errors='coerce')

# Set 'node' as index in hadera_demand_agg for efficient lookups
hadera_demand_agg.set_index('node', inplace=True)

# Display data types before the problematic line
display("gdf dtypes before update:", gdf[['TotalDemand', 'TotalTransfers']].dtypes)
display("hadera_demand_agg dtypes before update:", hadera_demand_agg[['Boardings_Hadera_sum', 'Alightings_Hadera_sum', 'TotalTransfers_Hadera_sum']].dtypes)


# Update TotalDemand and TotalTransfers in gdf using data from hadera_demand_agg
# Iterate through gdf and update rows where node matches in hadera_demand_agg
for index, row in gdf.iterrows():
    if row['node'] in hadera_demand_agg.index:
        # Get corresponding demand data from hadera_demand_agg
        hadera_data = hadera_demand_agg.loc[row['node']]

        # Update TotalDemand and TotalTransfers, adding to existing values
        gdf.loc[index, 'TotalDemand'] = gdf.loc[index, 'TotalDemand'] + hadera_data['Boardings_Hadera_sum'] + hadera_data['Alightings_Hadera_sum']
        gdf.loc[index, 'TotalTransfers'] = gdf.loc[index, 'TotalTransfers'] + hadera_data['TotalTransfers_Hadera_sum']


# Drop the aggregated columns from Hadera demand if they were added in a previous step
# Use a try-except block to handle cases where these columns might not exist
try:
    gdf.drop(columns=['Boardings_Hadera_sum', 'Alightings_Hadera_sum', 'TotalTransfers_Hadera_sum'], inplace=True)
except KeyError:
    pass # Columns not found, no need to drop


# Display the head of gdf to check the updated columns (optional)
# display(gdf.head())

In [ ]:
gdf.head(1)

# update this part, as 'Rail' is no longer a stand alone name, should use the 3 different kinds of 'Rail' or - read the end of the string where 'Rail' is

In [ ]:
gdf['area'].unique()

In [ ]:
demand.keys()

# update - no "Rail" in Mode Planned, but can contain Rail at the end.

In [ ]:
# for i, row in gdf[gdf['Mode_Planned'].str.contains('Rail')].iterrows():
for i, row in gdf.iterrows():
  if row['area'] in ['חיפה','צפון']:
    gdf.iloc[i, gdf.columns.get_loc('TotalDemand')] = demand['Haifa'][demand['Haifa']['Node']==row['node']]['Boardings_Daily'].sum() + demand['Haifa'][demand['Haifa']['Node']==row['node']]['Alightings_Daily'].sum()
    gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Haifa'][demand['Haifa']['Node']==row['node']]['TotalTransfers'].sum()
  elif row['area'] in ['תל אביב']:
    gdf.iloc[i, gdf.columns.get_loc('TotalDemand')] = demand['Tel Aviv'][demand['Tel Aviv']['Node']==row['node']]['Boardings_Daily'].sum() + demand['Tel Aviv'][demand['Tel Aviv']['Node']==row['node']]['Alightings_Daily'].sum()
    gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Tel Aviv'][demand['Tel Aviv']['Node']==row['node']]['TotalTransfers'].sum()
  elif row['area'] in ['באר שבע','דרום']:
    gdf.iloc[i, gdf.columns.get_loc('TotalDemand')] = demand['Beer Sheva'][demand['Beer Sheva']['Node']==row['node']]['Boardings_Daily'].sum() + demand['Beer Sheva'][demand['Beer Sheva']['Node']==row['node']]['Alightings_Daily'].sum()
    gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Beer Sheva'][demand['Beer Sheva']['Node']==row['node']]['TotalTransfers'].sum()
  elif row['area'] in ['ירושלים']:
    gdf.iloc[i, gdf.columns.get_loc('TotalDemand')] = demand['Jerusalem'][demand['Jerusalem']['Node']==row['node']]['Boardings_Daily'].sum() + demand['Jerusalem'][demand['Jerusalem']['Node']==row['node']]['Alightings_Daily'].sum()
    gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Jerusalem'][demand['Jerusalem']['Node']==row['node']]['TotalTransfers'].sum()

for i, row in gdf[gdf['Mode_Planned'].str.contains('Rail')].iterrows():
  if row['node'] in [400820, 400841]:
    gdf.iloc[i, gdf.columns.get_loc('TotalDemand')] = demand['Ashdod-Ashkelon'][demand['Ashdod-Ashkelon']['Node']==row['node']]['Boardings_Daily'].sum() + demand['Ashdod-Ashkelon'][demand['Ashdod-Ashkelon']['Node']==row['node']]['Alightings_Daily'].sum()
    gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Ashdod-Ashkelon'][demand['Ashdod-Ashkelon']['Node']==row['node']]['TotalTransfers'].sum()

In [ ]:
# only kiryat gat and ashkelon train stations have demand taken from ashdod-ashkelon, all other were taken from tel-aviv

In [ ]:
gdf[gdf['node']==511248]

# Update Shefaim LRT stop

In [ ]:
gdf.loc[gdf['node']==511248,'TotalDemand'] = 255.3

In [ ]:
gdf[gdf['node']==511248]

In [ ]:
gdf.to_csv('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8', index=False)

In [ ]:
gdf = gpd.read_file('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8')

In [ ]:
gdf['node'] = gdf['node'].astype('int64')

In [ ]:
# add 2050 employees & population in proximity to the h3_index
# first - group the hub list
# create centroid for the group, not the one h3_index.
# create 3 rings of influence - 0-600 meter (core), 600-1000 meter (primary) and 1000-1200 (extended)
# the whole 1.2 km is an influence area
# the scoring is weighted by the ring (1, 0.6 and 0.3)
# scoring should be normalized before weights


In [ ]:
# update Haifa Metronit using Haifa Metronit demand data

# Ensure 'Node' column in demand['Haifa Metronit'] is numeric
demand['Haifa Metronit']['Node'] = pd.to_numeric(demand['Haifa Metronit']['Node'], errors='coerce')

# Select relevant columns from demand['Haifa Metronit']
haifa_metronit_demand = demand['Haifa Metronit'][['Node', 'Boardings_Daily', 'Alightings_Daily', 'TotalTransfers']]

# Aggregate haifa_metronit_demand by Node, summing the demand and transfers
haifa_metronit_demand_agg = haifa_metronit_demand.groupby('Node').agg(
    Boardings_Haifa_Metronit_sum=('Boardings_Daily', 'sum'),
    Alightings_Haifa_Metronit_sum=('Alightings_Daily', 'sum'),
    TotalTransfers_Haifa_Metronit_sum=('TotalTransfers', 'sum')
).reset_index() # Reset index to make 'Node' a column again

# Rename 'Node' to 'node' for matching with gdf
haifa_metronit_demand_agg.rename(columns={'Node': 'node'}, inplace=True)

# Set 'node' as index in haifa_metronit_demand_agg for efficient lookups
haifa_metronit_demand_agg.set_index('node', inplace=True)

# Update TotalDemand and TotalTransfers in gdf using data from haifa_metronit_demand_agg
# Iterate through gdf and update rows where node matches in haifa_metronit_demand_agg
for index, row in gdf.iterrows():
    if row['node'] in haifa_metronit_demand_agg.index:
        # Get corresponding demand data from haifa_metronit_demand_agg
        haifa_metronit_data = haifa_metronit_demand_agg.loc[row['node']]

        # Update TotalDemand and TotalTransfers, adding to existing values
        gdf.loc[index, 'TotalDemand'] = gdf.loc[index, 'TotalDemand'] + haifa_metronit_data['Boardings_Haifa_Metronit_sum'] + haifa_metronit_data['Alightings_Haifa_Metronit_sum']
        gdf.loc[index, 'TotalTransfers'] = gdf.loc[index, 'TotalTransfers'] + haifa_metronit_data['TotalTransfers_Haifa_Metronit_sum']

# Display the head of gdf to check the updated columns
display(gdf.head())

In [ ]:
gdf[gdf['h3_index']=='8a3f4dca140ffff']

# Update specific nodes demand from National Model


*   Netanya
*   Netanya Sapir
*   Beit Yehoshua
*   Moshe Dayan (Rishon)
*   Modiin Merkaz
*   Modiin West



### Moshe Dayan

In [ ]:
gdf.loc[gdf['node']==400424, 'TotalDemand'] = 64985
gdf.loc[gdf['node']==400424, 'TotalTransfers'] = 43032

In [ ]:
gdf[gdf['group']=='777']

### Netanya

In [ ]:
gdf[gdf['node']==400020]

Because we don't have distinguish in the national model, regarding which mode_planned has the TotalDemand, we will set to one of the mode_planned, as long as they are the same node.

In [ ]:
gdf.loc[gdf.index==1057,:]

In [ ]:
gdf.loc[gdf.index==1057, 'TotalDemand'] = 108409
gdf.loc[gdf.index==1057, 'TotalTransfers'] = 84490

# update both, but only 1 will be taken, so there will be no mistakes
gdf.loc[gdf.index==1058, 'TotalDemand'] = 108409
gdf.loc[gdf.index==1058, 'TotalTransfers'] = 84490

In [ ]:
gdf[gdf['group']=='736']

### Netanya Sapir

In [ ]:
gdf[gdf['node']==400021]

In [ ]:
gdf[gdf['group']=='748']

In [ ]:
gdf.loc[gdf['node']==400021, 'TotalDemand'] = 23083
gdf.loc[gdf['node']==400021, 'TotalTransfers'] = 10140

### Beit Yehoshua

In [ ]:
gdf[gdf['node']==400030]

In [ ]:
gdf[gdf['group']=='749']

Because we have lrt here that is in the same group as the rail,<br>
we will adjust the total demand for the lrt node as well.

In [ ]:
# Rail
gdf.loc[gdf['node']==400030, 'TotalDemand'] = 14518
gdf.loc[gdf['node']==400030, 'TotalTransfers'] = 6101

# LRT
gdf.loc[gdf['node']==511246, 'TotalDemand'] = 13601
gdf.loc[gdf['node']==511246, 'TotalTransfers'] = 6101


### Modiin Merkaz

In [ ]:
gdf[gdf['node']==400470]

In [ ]:
gdf[gdf['group']=='311']

Because we don't have distinguish in the national model, regarding which mode_planned has the TotalDemand, we will set to one of the mode_planned, as long as they are the same node.

In [ ]:
gdf.loc[gdf.index==421, 'TotalDemand'] = 40628
gdf.loc[gdf.index==421, 'TotalTransfers'] = 0

gdf.loc[gdf.index==422, 'TotalDemand'] = 40628
gdf.loc[gdf.index==422, 'TotalTransfers'] = 0

### Modiin West

In [ ]:
gdf[gdf['node']==400460]

In [ ]:
gdf[gdf['group']=='310']

Because we don't have distinguish in the national model, regarding which mode_planned has the TotalDemand, we will set to one of the mode_planned, as long as they are the same node.

In [ ]:
gdf.loc[gdf.index==419, 'TotalDemand'] = 41000
gdf.loc[gdf.index==419, 'TotalTransfers'] = 12133

gdf.loc[gdf.index==420, 'TotalDemand'] = 41000
gdf.loc[gdf.index==420, 'TotalTransfers'] = 12133

In [ ]:
gdf.to_csv('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8', index=False)

In [ ]:
gdf = gpd.read_file('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8')

In [ ]:
gdf.head(1)

In [ ]:
gdf.info()

In [ ]:
gdf['Line_Nunique'] = pd.to_numeric(gdf['Line_Nunique'], errors='coerce').fillna(0)
gdf['TotalDemand'] = pd.to_numeric(gdf['TotalDemand'], errors='coerce').fillna(0)
gdf['TotalTransfers'] = pd.to_numeric(gdf['TotalTransfers'], errors='coerce').fillna(0)

In [ ]:
grouped_hubs.columns

In [ ]:
grouped_hubs[grouped_hubs['group']==583][['TotalDemand','address','node']]

In [ ]:
grouped_hubs.sort_values(by='LRT Lines', ascending=False)

In [ ]:
grouped_hubs.to_csv('/content/drive/MyDrive/Hubs/Grouped_Hubs_ReadyForPopEmp_29102025.csv', encoding='utf-8')

In [ ]:
grouped_hubs[grouped_hubs['group']==583][['TotalDemand','TotalTransfers']]

In [ ]:
node_demand_agg[node_demand_agg['node']=='400080']

# Glilot Darom - metro and rail far apart, need to adjust group id so they will be together.

In [ ]:
grouped_hubs[grouped_hubs['group']==576]

In [ ]:
grouped_hubs.loc[grouped_hubs['group']==577, 'group'] = 576

In [ ]:
gdf.loc[gdf['group']==577, 'group'] = 576

In [ ]:
gdf[gdf['group']==576]

Correct Code for grouping

In [ ]:
# Ensure 'group' column is of integer type for consistent grouping
gdf['group'] = pd.to_numeric(gdf['group'], errors='coerce').fillna(-1).astype(int)

# Aggregate TotalDemand and TotalTransfers per unique node within each group *before* dissolving
node_demand_agg = gdf.groupby(['group', 'node']).agg({
    'TotalDemand': 'first',
    'TotalTransfers': 'first'
}).reset_index()

# Create a dictionary mapping group IDs to their total demand and transfers
group_demand_lookup = node_demand_agg.groupby('group').agg({
    'TotalDemand': 'sum',
    'TotalTransfers': 'sum'
}).to_dict('index')

# First, create aggregated columns in the original dataframe for the dissolve operation
def get_group_total_demand(group_id):
    return group_demand_lookup.get(group_id, {}).get('TotalDemand', 0)

def get_group_total_transfers(group_id):
    return group_demand_lookup.get(group_id, {}).get('TotalTransfers', 0)

# Add the aggregated totals as new columns
gdf['GroupTotalDemand'] = gdf['group'].apply(get_group_total_demand)
gdf['GroupTotalTransfers'] = gdf['group'].apply(get_group_total_transfers)

# Group the gdf by the 'group' column and dissolve geometries using dissolve()
grouped_hubs = gdf.dissolve(by='group', aggfunc={
    'h3_index': list,  # Keep all h3_index values as a list
    'node': lambda x: list(x), # Aggregate nodes into a list
    'Mode_Planned': lambda x: list(x.unique()), # Aggregate unique Mode_Planned into a list
    'Model': lambda x: list(x.unique()), # Aggregate unique Models into a list
    # Modified Line_Unique aggregation to return a flat list of unique lines
    'Line_Unique': lambda x: list(set([item for sublist in x for item in (sublist if isinstance(sublist, (list, np.ndarray)) else [sublist])])),
    'address': lambda x: x.iloc[0] if not x.isnull().all() else None, # Keep the first non-null address
    'area': lambda x: list(x.unique()), # Aggregate unique areas into a list
    'location': lambda x: list(x.unique()), # Aggregate unique locations into a list
    # Use the pre-calculated group totals (these will be the same for all rows in each group)
    'GroupTotalDemand': 'first',
    'GroupTotalTransfers': 'first'
}).reset_index() # Reset index to make 'group' a column

# Rename the columns to the desired names
grouped_hubs = grouped_hubs.rename(columns={
    'GroupTotalDemand': 'TotalDemand',
    'GroupTotalTransfers': 'TotalTransfers'
})

# Convert the result to a GeoDataFrame (dissolve already returns a GeoDataFrame, but good practice to be explicit)
grouped_hubs = gpd.GeoDataFrame(grouped_hubs, geometry='geometry', crs=gdf.crs)

# Calculate the number of unique lines per group based on the aggregated 'Line_Unique' list
grouped_hubs['Total_Unique_Lines'] = grouped_hubs['Line_Unique'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Calculate sum of Line_Nunique per Mode_Planned per group
# This part remains useful for seeing the breakdown by mode within the group
line_nunique_by_mode = gdf.groupby(['group', 'Mode_Planned'])['Line_Nunique'].sum().unstack(fill_value=0)

# Rename columns to desired format (e.g., 'BRT_Line_Nunique')
line_nunique_by_mode.columns = [f'{col} Lines' for col in line_nunique_by_mode.columns]

# Merge the new Line_Nunique columns back into grouped_hubs
grouped_hubs = grouped_hubs.merge(line_nunique_by_mode, left_on='group', right_index=True, how='left')

# Create centroid for each group's geometry
# Ensure the GeoDataFrame is in a projected CRS before calculating centroids for accurate results
if grouped_hubs.crs.is_geographic:
    grouped_hubs = grouped_hubs.to_crs(epsg=crs_il) # Reproject to a projected CRS for Israel

grouped_hubs['centroid'] = grouped_hubs.geometry.centroid

# Clean up the temporary columns from the original dataframe
gdf = gdf.drop(columns=['GroupTotalDemand', 'GroupTotalTransfers'])

# Display the head of the new grouped_hubs GeoDataFrame
display(grouped_hubs.head())

In [ ]:
grouped_hubs[grouped_hubs['group']==576]

# correct: HighSpeed Rail in Savidor!

In [ ]:
grouped_hubs.columns

In [ ]:
grouped_hubs.to_csv('/content/drive/MyDrive/Hubs/Grouped_Hubs_ReadyForPopEmp_29102025.csv', encoding='utf-8')

In [ ]:
grouped_hubs = pd.read_csv('/content/drive/MyDrive/Hubs/Grouped_Hubs_ReadyForPopEmp_29102025.csv', encoding='utf-8')

In [ ]:
grouped_hubs[grouped_hubs['group']==955]

In [ ]:
grouped_hubs[grouped_hubs['group']==955]['Line_Unique'].values

In [ ]:
grouped_hubs[grouped_hubs['group']==955]['Mode_Planned'].values

In [ ]:
grouped_hubs[grouped_hubs['group']==921]['Line_Unique'].values

In [ ]:
grouped_hubs[grouped_hubs['group']==920]['Line_Unique'].values

In [ ]:
# Load the CSV file
import folium
from shapely import wkt
import geopandas as gpd
import pandas as pd

all_nodes_file = '/content/drive/MyDrive/Hubs/All_nodes+lines_21082025.csv'
encoding = 'windows-1255'
crs_il = 2039

all_nodes_df = pd.read_csv(all_nodes_file, encoding=encoding)

# Convert to GeoDataFrame, assuming 'geometry' column is in WKT format
# Use errors='coerce' to handle invalid WKT strings
all_nodes_df['geometry'] = all_nodes_df['geometry'].apply(lambda geom: wkt.loads(geom) if isinstance(geom, str) else None)
all_nodes_gdf = gpd.GeoDataFrame(all_nodes_df, geometry='geometry', crs=crs_il)

# Reproject to WGS84 (EPSG:4326) for Folium
all_nodes_gdf_wgs84 = all_nodes_gdf.to_crs(epsg=4326)

# Create a Folium map centered around the data
# Filter out rows with None geometry before calculating the centroid of the union
valid_geometries = all_nodes_gdf_wgs84.dropna(subset=['geometry']).geometry
map_center = valid_geometries.unary_union.centroid.coords[0][::-1]
m = folium.Map(location=map_center, zoom_start=8)

# Add markers to the map with tooltips
for index, row in all_nodes_gdf_wgs84.iterrows():
    if row.geometry is not None: # Check if geometry is not None after loading
        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            tooltip=f"LINE_ID: {row['LINE_ID']}<br>Node: {row['node']}"
        ).add_to(m)

# Display the map
m